In [8]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

JupyterDash.infer_jupyter_proxy_config()

# OS + data tools
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# CRUD Module
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "aacpassword"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([

    html.Center([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            height=120
        ),
        html.H3("Tristin Raymond – Grazioso Salvare Dashboard")
    ]),

    html.Hr(),

    # Filter Controls
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All Animals', 'value': 'ALL'},
                {'label': 'Water Rescue', 'value': 'WATER'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'MOUNTAIN'},
                {'label': 'Disaster/Tracking', 'value': 'DISASTER'}
            ],
            value='ALL',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),

    html.Hr(),

    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True}
                 for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={'overflowX': 'auto'}
    ),

    html.Br(),
    html.Hr(),

    # Graph + Map Layout
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ]
    )
])


#############################################
# Controller / Callbacks
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    if filter_type == 'WATER':
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Labrador|Chesapeake"}
        }

    elif filter_type == 'MOUNTAIN':
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "German Shepherd|Alaskan"}
        }

    elif filter_type == 'DISASTER':
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Bloodhound|Rottweiler"}
        }

    else:
        query = {}

    df = pd.DataFrame.from_records(db.read(query))
    df.drop(columns=['_id'], inplace=True)

    return df.to_dict('records')


# Pie Chart
@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data")
)
def update_graphs(viewData):

    if viewData is None:
        return []

    dff = pd.DataFrame(viewData)

    fig = px.pie(
        dff,
        names='breed',
        title='Animal Breeds Distribution'
    )

    return [dcc.Graph(figure=fig)]


# Highlight selected column
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Map Update
@app.callback(
    Output('map-id', "children"),
    Input('datatable-id', "derived_virtual_data"),
    Input('datatable-id', "derived_virtual_selected_rows")
)
def update_map(viewData, index):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run Dashboard
app.run_server()

Dash app running on https://beginfinal-virtualmanual-3000.codio.io/proxy/8050/
